# T08 — Protocolo walk-forward comum

Script: `walk_forward.py`. Vale para os 20 pipelines (4 modelos × 5 bases).

- **Corte:** cronológico, sem embaralhamento — Bitcoin 70/30; tráfego, poluição e clima 80/20; ouro 75/25 (`inicio_teste` de `dados_tratados`).
- **Horizonte:** 1 passo (dia, hora ou semana).
- **Origens:** datas do teste com alvo e features completos (`preparar_features_base`); iguais para todos os modelos da base.
- **Causalidade:** features só até `t-1`; escalonadores e hiperparâmetros ajustados só no histórico; teste nunca usado para escolher parâmetros.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'walk_forward.py').is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from walk_forward import BASES, POLITICAS, protocolo, validar_previsoes


## Protocolo por base

In [ ]:
tabela_protocolo = pd.DataFrame([protocolo(b, PROJECT_ROOT) for b in BASES])
display(tabela_protocolo)


## Política de treino por modelo

In [ ]:
display(pd.DataFrame(POLITICAS.items(), columns=['modelo', 'politica']))


## Conferência das previsões salvas

Lê os CSVs de previsão que cada notebook grava e confere se as datas são exatamente as origens oficiais. Arquivos ainda não gerados aparecem como `ausente`.


In [ ]:
ARQUIVOS = {
    ('clima', 'random_forest'): 'resultados_random_forest/previsoes_random_forest_grupo4.csv',
    ('clima', 'sarimax'): 'resultados_sarimax/previsoes_sarimax_grupo4.csv',
    ('clima', 'mlp'): 'resultados_mlp/previsoes_mlp_grupo4.csv',
}
COLUNAS_DATA = ('data', 'date', 'datetime', 'timestamp')

linhas = []
for (base, modelo), relativo in ARQUIVOS.items():
    caminho = PROJECT_ROOT / relativo
    if not caminho.is_file():
        linhas.append({'base': base, 'modelo': modelo, 'status': 'ausente'})
        continue
    prev = pd.read_csv(caminho)
    coluna = next(c for c in prev.columns if c.lower() in COLUNAS_DATA)
    r = validar_previsoes(base, prev[coluna], PROJECT_ROOT)
    linhas.append({'modelo': modelo, **r, 'status': 'ok' if r['ok'] else 'divergente'})
display(pd.DataFrame(linhas))


## Limitações registradas

- **SARIMAX (grupo 4):** recebe a sequência filtrada como arrays; lacunas na grade (ex.: ~10 dias antes de 04/11/2016) contam como um único passo. As datas avaliadas coincidem com RF e MLP, mas a dinâmica do estado é afetada.
- **SARIMAX:** o número de reajustes vem do orçamento de tempo da máquina (8 blocos na execução salva); outra máquina pode gerar outro protocolo.
- **RF/MLP × SARIMAX:** políticas diferentes (modelo fixo × reajuste periódico), ambas causais; declarar na comparação de MAE.
- **Holt-Winters do grupo 1:** atualmente mensal, horizonte 12 meses; fora deste protocolo até ser adaptado.
